# RAG pipeline

In [4]:
import os
import sys
from dotenv import load_dotenv
from llama_index.core import (
    Document,
    VectorStoreIndex,
    StorageContext,
    PromptTemplate,
    load_index_from_storage,
    set_global_handler
)
from llama_index.llms.vllm import Vllm
from llama_index.llms.google_genai import GoogleGenAI
from llama_index.core.node_parser import SentenceSplitter
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.settings import Settings
from llama_cloud import AsyncLlamaCloud

# logging
import logging
logging.basicConfig(stream=sys.stdout, level=logging.INFO)
logging.getLogger("httpcore").setLevel(logging.WARNING)
logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("llama_index.core").setLevel(logging.WARNING)
logging.getLogger("fsspec").setLevel(logging.WARNING)
set_global_handler("simple")

load_dotenv()
client = AsyncLlamaCloud(api_key=os.getenv("LLAMA_CLOUD_API_KEY"))

## Setup LLM and Embedding model

* For the LLM, use **Gemini** for local development and *vLLM* for production
* For the embedding model use **BAAI/bge-small-en-v1.5** for local and Qwen3

In [ ]:
if os.getenv("APP_ENV") == "dev":
    embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")
    gemini = GoogleGenAI(model="gemini-2.5-flash-lite")
elif os.getenv("APP_ENV") == "prod":
    embed_model = HuggingFaceEmbedding(model_name="Qwen/Qwen3-Embedding-0.6B")
    llm = Vllm(
        model="Qwen3.5-122B-A10B-GGUF",
        tensor_parallel_size=4,
        max_new_tokens=100,
        vllm_kwargs={"swap_space": 1, "gpu_memory_utilization": 0.5},
    )

Settings.llm = llm
Settings.embed_model = embed_model

INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: BAAI/bge-small-en-v1.5


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5557.00it/s]
BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


INFO:sentence_transformers.SentenceTransformer:1 prompt is loaded, with the key: query


# Parse the PDF

In [9]:
async def parse_documents_with_llamaparse(data_dir: str):
    documents = []

    for filename in os.listdir(data_dir):
        if not filename.endswith(".pdf"):
            continue

        file_path = os.path.join(data_dir, filename)

        print("Uploading file to LlamaCloud...")
        file_obj = await client.files.create(
            file=file_path,
            purpose="parse"
        )

        print("Parsing file...")
        result = await client.parsing.parse(
            file_id=file_obj.id,
            tier="cost_effective",
            version="latest",
            agentic_options={
                "custom_prompt": "This is an equipment manual. Pay special attention to technical specifications, limitations and warnings."
            },
            output_options={
                "markdown": {
                    "tables": {
                        "output_tables_as_markdown": True,
                    },
                },
                "images_to_save": ["embedded"],
            },
            expand=["markdown"]
        )

        print(result.job.status)
        print(result.markdown)
        print(f"Extracting pages...")
        for page in result.markdown.pages:
            documents.append(
                Document(
                    text=page.markdown,
                    metadata={
                        "file_name": filename,
                        "page": page.page_number
                    }
                )
            )

    return documents

documents = await parse_documents_with_llamaparse("../data")

Uploading file to LlamaCloud...
Parsing file...
COMPLETED
Markdown(pages=[MarkdownPageMarkdownResultPage(markdown='# SM2 Series SE\n\n# Smart Camera User Manual\n\n# V2.5.4. 2025\n\nwww.contrastech.com', page_number=1, success=True, footer=None, header=None), MarkdownPageMarkdownResultPage(markdown='# Preface\n\nSoftware Operation\n\n# SM-Datum Operation\n\nDouble-click the SM-Datum shortcut on the desktop to open up the client software. The Software refreshes the camera list automatically. To add a remote camera, click and type in the IP address.\n\n# Purpose\n\nMake sure the connection between the remote camera and the PC is established when adding remote cameras.\n\nThis Manual is a basic description of SM2 series Smart Cameras, which mainly includes the product description, quick installation guide and Simple introduction of SDK(SM-Datum). This manual may be updated due to product upgrades or other reasons. If you need, please contact the sales engineer for the latest version of th

## Chunk the document and store the RAG index

NOTE: If the parsing or chunking strategy is changed it will NOT automatically update your existing stored index. It loads the old index with old chunks
* After applying a new parsing or chunking strategy, delete the old storage folder.

In [10]:
# chunking
splitter = SentenceSplitter(
    chunk_size=512,
    chunk_overlap=50
)

print("Chunking documents into nodes...")
nodes = splitter.get_nodes_from_documents(documents)

# vector index
if os.path.exists("../storage"):
    print("Loading index from storage...")
    storage_context = StorageContext.from_defaults(persist_dir="../storage")
    index = load_index_from_storage(storage_context)
else:
    print("Creating new index...")
    index = VectorStoreIndex.from_documents(nodes)
    index.storage_context.persist("../storage")

Chunking documents into nodes...
Creating new index...


In [11]:
# basic retriever
retriever = index.as_retriever(similarity_top_k=5)

print("Querying the index...")
query_engine = RetrieverQueryEngine.from_args(
    retriever,
    llm=llm
)

print("Generating response...")
response = await query_engine.aquery(
    "What is the name of this device?"
)
print(response)

Querying the index...
Generating response...
INFO:google_genai.models:AFC is enabled with max remote calls: 10.
** Messages: **
system: You are an expert Q&A system that is trusted around the world.
Always answer the query using the provided context information, and not prior knowledge.
Some rules to follow:
1. Never directly reference the given context in your answer.
2. Avoid statements like 'Based on the context, ...' or 'The context information ...' or anything along those lines.
user: Context information is below.
---------------------
file_name: User_Manual_SM2_SE_Camera_CT_Ver.2.5.4_EN.pdf
page: 6

Equipment shown in Fig. 1-2

Double-click the SM-Datum shortcut on the desktop to open up the client software.

| Color  | Pin | Signal    | Signal Source | Designation                                                                    | Cable type               |
| ------ | --- | --------- | ------------- | -----------------------------------------------------------------------------

# RAG Evaluation

In [ ]:
eval_dataset = [
    {
        "question": "Can I use the 12V power plug open line and DB9 female serial port connector at the same time.?",
        "ground_truth": "No, it may damage the power supply"
    },
    {
        "question": "How many indicators are there on the side of the device that show the device status?",
        "ground_truth": "There are three indicators."
    }
]

In [13]:
def evaluate_retrieval(retriever, eval_dataset):
    results = []

    for sample in eval_dataset:
        nodes = retriever.retrieve(sample["question"])
        contexts = [n.text for n in nodes]

        hit = any(sample["ground_truth"].lower() in c.lower() for c in contexts)

        results.append({
            "question": sample["question"],
            "hit": hit,
            "contexts": contexts
        })

    hit_rate = sum(r["hit"] for r in results) / len(results)
    print(f"Retrieval Hit Rate: {hit_rate:.2f}")

    return results

In [14]:
judge_prompt = PromptTemplate(
    """You are evaluating an answer.

Question: {question}
Ground Truth: {ground_truth}
Answer: {answer}

Is the answer acceptable? Respond with ONLY 'yes' or 'no'.
"""
)

async def evaluate_with_llm(query_engine, eval_dataset, llm):
    scores = []

    for sample in eval_dataset:
        response = await query_engine.aquery(sample["question"])
        answer = str(response)

        judge_input = judge_prompt.format(
            question=sample["question"],
            ground_truth=sample["ground_truth"],
            answer=answer
        )

        judgment = await llm.acomplete(judge_input)
        is_correct = "yes" in judgment.text.lower()

        scores.append(is_correct)

    accuracy = sum(scores) / len(scores)
    print(f"LLM Judged Accuracy: {accuracy:.2f}")

In [15]:
evaluate_retrieval(retriever, eval_dataset)
await evaluate_with_llm(query_engine, eval_dataset, llm)

Retrieval Hit Rate: 1.00
INFO:google_genai.models:AFC is enabled with max remote calls: 10.
** Messages: **
system: You are an expert Q&A system that is trusted around the world.
Always answer the query using the provided context information, and not prior knowledge.
Some rules to follow:
1. Never directly reference the given context in your answer.
2. Avoid statements like 'Based on the context, ...' or 'The context information ...' or anything along those lines.
user: Context information is below.
---------------------
file_name: User_Manual_SM2_SE_Camera_CT_Ver.2.5.4_EN.pdf
page: 6

|
| Gray  | 6   | DI\_0     | It can be configured as input or output, and is input by default. |
| Black | 7   | GND       | DC Power Supply Negative                                          |
| Red   | 8   | POWER\_IN | DC Power Supply Positive                                          |

You cannot use DB9 female serial port and VCC to power the device at the same time. Otherwise, damaging to power sup